# Step 23b — Correct per-pair committors, a floor for committor fidelity, and like-for-like location tests (CPU)

**Inputs.** Mount the committed outputs of **Step 19** (`step19_logits.npz`) and **Step 23** (`step23_committor.npz`, `step23_conditional.npz`). CPU only, about 45 minutes.

1. **Per-pair committor decisions without selection bias.** For a pair $(a,b)$, take the trajectories that end in $a$ or $b$. At each branching level, pool their continuations that land in $\{a,b\}$ and compute the cross-over rate, the fraction landing on the *other* class of the pair. The pair is decided below the noise level from which this rate stays under $0.1$. This uses no per-state eligibility filter, and parent bootstrap intervals are reported. A sanity check follows: animal/animal pairs must not be decided before animal-versus-vehicle.
2. **Committor fidelity against its floor.** At low noise, the distance between the model's committor and the classifier posterior is dominated by the classifier's own errors (86% versus the ViT's 98%). We subtract that plateau and report the transition-region excess with parent-bootstrap intervals, plus a per-level calibration slope.
3. **Location, like for like.** Bootstrap-median transitions at $n=4000$ for: the conditional model; the unconditional model raw and rebalanced; and train-versus-test data, which gives the floor at this $n$. Reported as scatter, and as median offset with class-block intervals.
4. **The residual offset of the rebalanced unconditional model** at full size ($n=14500$), with a class-block interval.

In [ ]:
import glob, numpy as np, matplotlib.pyplot as plt, time
from itertools import combinations
from scipy.signal import fftconvolve
from scipy.stats import spearmanr, rankdata
plt.rcParams['figure.dpi'] = 120; rng = np.random.default_rng(23)
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']; VEH = {0, 1, 8, 9}; PAIRS = list(combinations(range(10), 2))
kd = lambda p: 'va' if ((p[0] in VEH) != (p[1] in VEH)) else ('vv' if p[0] in VEH else 'aa')
def find(pat): p = glob.glob(f'/kaggle/input/**/{pat}', recursive=True); assert p, f'mount the output containing {pat}'; return p[0]
Z19 = np.load(find('step19_logits.npz')); L_TE, W_TE, L_TR, W_TR, L_M, W_M, SIG = [Z19[k] for k in ['L_TE', 'W_TE', 'L_TR', 'W_TR', 'L_M', 'W_M', 'SIG_REC']]
ZC = np.load(find('step23_committor.npz')); END, PAR, Q_CLF, LEV_SIG = ZC['END'], ZC['PAR'], ZC['Q_CLF'], ZC['LEV_SIG']; L_LEV, M_PAR, K_BR = END.shape
ZK = np.load(find('step23_conditional.npz')); L_C, W_C = ZK['L'], ZK['W']
print('committor', END.shape, 'conditional', W_C.shape, 'unconditional', W_M.shape)
G = np.linspace(-4.0, 4.0, 4001); DGG = G[1]-G[0]
def wkde_score(m, w):
    m = np.clip(m, -8, 8); w = np.asarray(w, dtype=np.float64); neff = w.sum()**2/np.maximum((w**2).sum(), 1e-12)
    mu = np.average(m, weights=w); sd = np.sqrt(np.average((m-mu)**2, weights=w)); bw = max(1.5*1.06*sd*neff**(-1/5), 1e-3)
    h, _ = np.histogram(m, bins=np.append(G-DGG/2, G[-1]+DGG/2), weights=w, density=True); k = np.exp(-G**2/(2*bw**2)); k /= k.sum()
    p = np.maximum(fftconvolve(h, k, 'same'), 1e-300); return np.gradient(np.log(p), G), p
def _merge(pts, merge=0.03):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts
def stationary(sm, p, lo=-3.5, hi=3.5, prominence=0.05):
    keep = (G > lo) & (G < hi) & (p > 1e-3*p.max()); idx = np.where((np.sign(sm[:-1]) != np.sign(sm[1:])) & keep[:-1] & keep[1:])[0]
    pts = _merge([(float(G[i]-sm[i]*DGG/(sm[i+1]-sm[i])) if sm[i+1] != sm[i] else float(G[i]), 'max' if sm[i] > 0 else 'min') for i in idx])
    logp = np.concatenate([[0.0], np.cumsum(0.5*(sm[1:]+sm[:-1])*DGG)]); lp = lambda x: float(np.interp(x, G, logp)); ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if pts[a][1] != pts[a+1][1] and abs(lp(pts[a][0]) - lp(pts[a+1][0])) < prominence: del pts[a:a+2]; ch = True; break
    return pts
def n_max(pts): return sum(k == 'max' for _, k in pts)
def cusp(sm, p, window=1.0, b_min=0.02, s_max=0.6):
    ok = (np.abs(G) < window) & (p > 0); c3, c2, c1, c0 = np.polyfit(G[ok], sm[ok], 3, w=np.sqrt(p[ok])); b = max(-c3, b_min); c3 = -b; s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    return c0 + c1*s + c2*s**2 + c3*s**3, c1 + 3*c3*s**2 + 2*c2*s, b
def std_stats(z, wa, wb):
    mua, mub = np.average(z, weights=wa), np.average(z, weights=wb); half = 0.5*(mua-mub); sda = np.sqrt(np.average((z-mua)**2, weights=wa)); sdb = np.sqrt(np.average((z-mub)**2, weights=wb))
    return 0.5*(mua+mub), half, abs(half)/(0.5*(sda+sdb))
STD = {(a, b): [std_stats(L_TE[g, :, a].astype(np.float64) - L_TE[g, :, b], W_TE[:, a] + 1e-12, W_TE[:, b] + 1e-12) for g in range(len(SIG))] for a, b in PAIRS}
def analyse(L, W, a, b, idx, balance=False, rw=None, g0=0, slope=False):
    """Along the pair log-odds (reference standardisation): returns dict(sigma, gi, h, a, b[, s]). rw: extra class reweighting factors (for part C)."""
    Wa = W[idx, a].astype(np.float64); Wb = W[idx, b].astype(np.float64)
    if rw is not None: Wa = Wa*rw[a]; Wb = Wb*rw[b]
    if balance: tot = 0.5*(Wa.sum()+Wb.sum()); Wa = Wa/Wa.sum()*tot; Wb = Wb/Wb.sum()*tot
    def dist(g):
        mid, half, sep = STD[(a, b)][g]
        if sep < 0.3: return None
        z = L[g, idx, a].astype(np.float64) - L[g, idx, b]; return wkde_score((z - mid)/half, Wa + Wb)
    prev = None if g0 == 0 else 1
    for g in range(g0, len(SIG)):
        d = dist(g); nm = 1 if d is None else n_max(stationary(*d))
        if nm >= 2 and prev is not None:
            h, a_, b_ = cusp(*d); out = dict(sigma=float(SIG[g]), gi=g, h=h, a=a_, b=b_)
            if slope:                                            # a(log sigma) near the transition, from the levels just above and below
                gs = [k for k in range(max(0, g-4), min(len(SIG), g+3)) if dist(k) is not None]
                av = [cusp(*dist(k))[1] for k in gs]
                if len(gs) >= 3: out['s'] = -np.polyfit(np.log(SIG[gs]), av, 1)[0]
            return out
        if nm == 1: prev = 1
    return None
def boot(L, W, a, b, n, B, balance=False, ref=False, g_hint=None):
    """Bootstrap medians and 95% intervals of sigma_bif and h at sample size n."""
    ss, hs = [], []; g0 = 0 if g_hint is None else max(0, g_hint - 8)
    for _ in range(B):
        idx = rng.choice(len(W), size=n, replace=True); r = analyse(L, W, a, b, idx, balance, g0=g0)
        if r: ss.append(r['sigma']); hs.append(r['h'])
    if len(ss) < B//3: return None
    return dict(sigma=np.median(ss), s_lo=np.percentile(ss, 2.5), s_hi=np.percentile(ss, 97.5), h=np.median(hs), h_lo=np.percentile(hs, 2.5), h_hi=np.percentile(hs, 97.5), n_ok=len(ss))
def class_block_ci(x, y, pairs, B=5000, fn=None):
    """Class-block bootstrap: resample the 10 classes with replacement, weight each pair by the product of its classes' multiplicities."""
    P = np.array(pairs); out = []
    for _ in range(B):
        cls = rng.choice(10, 10, replace=True); w = np.array([np.sum(cls == a)*np.sum(cls == b) for a, b in P], float)
        if w.sum() < 6: continue
        if fn is None:
            mx, my = np.average(x, weights=w), np.average(y, weights=w); out.append(np.sum(w*(x-mx)*(y-my))/np.sqrt(np.sum(w*(x-mx)**2)*np.sum(w*(y-my)**2)))
        else: out.append(fn(w))
    out = np.array(out, float); out = out[np.isfinite(out)]
    return np.percentile(out, [2.5, 97.5]) if len(out) > 20 else np.array([np.nan, np.nan])

def weighted_spearman_ci(x, y, pairs, B=4000):
    rx, ry = rankdata(x), rankdata(y); return class_block_ci(rx, ry, pairs, B=B)
def median_offset_ci(lr, pairs, B=4000):
    return class_block_ci(None, None, pairs, B=B, fn=lambda w: np.exp(np.average(lr, weights=w)))
print('ready')

## 1. Per-pair committor decisions (pooled cross-over rate)

In [ ]:
def crossover(a, b, J, min_count=20):
    X = np.full(L_LEV, np.nan)
    for l in range(L_LEV):
        same = sum(np.sum(END[l, j] == PAR[j]) for j in J); cross = sum(np.sum(END[l, j] == (b if PAR[j] == a else a)) for j in J)
        if same + cross >= min_count: X[l] = cross/(same + cross)
    return X
def decide(X, thr=0.1):
    """Highest noise level from which the cross-over rate stays <= thr (levels are in descending sigma), log-interpolated at the crossing."""
    ok = np.isfinite(X); good = ok & (X <= thr)
    for l in range(L_LEV):
        if good[l] and all(good[k] or not ok[k] for k in range(l, L_LEV)):
            if l > 0 and ok[l-1] and X[l-1] > thr:
                w = (X[l-1] - thr)/(X[l-1] - X[l]); return float(np.exp(np.log(LEV_SIG[l-1]) + w*(np.log(LEV_SIG[l]) - np.log(LEV_SIG[l-1]))))
            return float(LEV_SIG[l])
    return np.nan
DEC, DLO, DHI = {}, {}, {}
for a, b in PAIRS:
    J = np.where(np.isin(PAR, [a, b]))[0]
    if len(J) < 20: continue
    d = decide(crossover(a, b, J)); bs = [decide(crossover(a, b, rng.choice(J, len(J)))) for _ in range(200)]
    if np.isfinite(d): DEC[(a, b)] = d; DLO[(a, b)], DHI[(a, b)] = np.nanpercentile(bs, [2.5, 97.5])
isv = np.isin(PAR, list(VEH)); same_sup = (np.isin(END, list(VEH)) == isv[None, :, None]).mean(2); same_cls = (END == PAR[None, :, None]).mean(2)
def dec_frac(frac, thr=11/12):
    out = np.full(frac.shape[1], np.nan)
    for j in range(frac.shape[1]):
        t = np.flip(np.cumprod(np.flip(frac[:, j] >= thr))).astype(bool)
        if t.any(): out[j] = LEV_SIG[np.argmax(t)]
    return out
dsup, dcls = dec_frac(same_sup), dec_frac(same_cls)
print(f"pooled per-pair decisions for {len(DEC)} pairs (cross-over rate <= 0.1)")
for k in ['va', 'vv', 'aa']:
    v = [DEC[p] for p in DEC if kd(p) == k]; print(f"   {k}: median pair decision {np.median(v):.2f} (n={len(v)})")
print(f"sanity: superclass decided below {np.nanmedian(dsup):.2f}; class below {np.nanmedian(dcls):.2f} (animals {np.nanmedian(dcls[~isv]):.2f}, vehicles {np.nanmedian(dcls[isv]):.2f})")
print("   animal/animal pair decisions should not exceed the superclass decision; if they do, the estimator is still biased.")
# like-for-like: pair decision vs pairwise density transition (model and data), class-block intervals
n = len(W_M); rows = []
for p, d in DEC.items():
    a, b = p; ir = rng.choice(len(W_TE), size=min(n, len(W_TE)), replace=False); rd = analyse(L_TE, W_TE, a, b, ir); rm = analyse(L_M, W_M, a, b, np.arange(n))
    if rd and rm: rows.append((p, d, rd['sigma'], rm['sigma']))
P = [r[0] for r in rows]; dd = np.array([r[1] for r in rows]); sdat = np.array([r[2] for r in rows]); smod = np.array([r[3] for r in rows])
for nm, s in [('model sigma_bif', smod), ('data sigma_bif', sdat)]:
    ci = weighted_spearman_ci(dd, s, P); print(f"Spearman(pair decision, {nm}) = {spearmanr(dd, s).correlation:.2f}, class-block 95% CI [{ci[0]:.2f}, {ci[1]:.2f}]")
for k in ['va', 'vv', 'aa']:
    j = [i for i, p in enumerate(P) if kd(p) == k]; print(f"   {k}: decision/sigma_bif(model) median {np.median(dd[j]/smod[j]):.2f} (IQR {np.percentile(dd[j]/smod[j],25):.2f}-{np.percentile(dd[j]/smod[j],75):.2f})")
ci = median_offset_ci(np.log(dd/smod), P); print(f"overall decision/sigma_bif(model): median {np.median(dd/smod):.2f}; geometric mean {np.exp(np.mean(np.log(dd/smod))):.2f}, class-block 95% CI [{ci[0]:.2f}, {ci[1]:.2f}]")

## 2. Committor fidelity above the classifier floor

In [ ]:
Q_MOD = np.stack([np.stack([np.bincount(END[l, m], minlength=10)/K_BR for m in range(M_PAR)]) for l in range(L_LEV)])
rs = np.random.default_rng(5)
def excess(idx):
    out = []
    for l in range(L_LEV):
        tv = 0.5*np.abs(Q_MOD[l, idx] - Q_CLF[l, idx]).sum(1).mean()
        sim = np.mean([0.5*np.abs(np.bincount(rs.choice(10, K_BR, p=q/q.sum()), minlength=10)/K_BR - q).sum() for q in Q_CLF[l, idx]])
        out.append(tv - sim)
    return np.array(out)
ex0 = excess(np.arange(M_PAR)); floor0 = ex0[-2:].mean()
BS = np.array([excess(rng.choice(M_PAR, M_PAR)) for _ in range(100)]); fl = BS[:, -2:].mean(1, keepdims=True); above = BS - fl
print(f"{'sigma':>7} {'excess over 12-draw noise':>26} {'above low-noise floor [95% CI]':>34} {'calibration slope':>18}")
for l, s in enumerate(LEV_SIG):
    x = Q_CLF[l].ravel(); y = Q_MOD[l].ravel(); sl = np.polyfit(x, y, 1)[0]
    print(f"{s:7.3f} {ex0[l]:26.3f} {ex0[l]-floor0:+12.3f} [{np.percentile(above[:, l], 2.5):+.3f}, {np.percentile(above[:, l], 97.5):+.3f}] {sl:18.2f}")
print(f"low-noise floor (classifier's own error) = {floor0:.3f}")

## 3–4. Location like for like, and the residual offsets

In [ ]:
def boot_sigma(L, W, a, b, n, B=60, balance=False):
    r0 = analyse(L, W, a, b, rng.choice(len(W), size=n, replace=len(W) < n), balance)
    if r0 is None: return None
    ss = []
    for _ in range(B):
        r = analyse(L, W, a, b, rng.choice(len(W), size=n, replace=True), balance, g0=max(0, r0['gi'] - 8))
        if r: ss.append(r['sigma'])
    return np.median(ss) if len(ss) >= B//3 else None
def location_set(n):
    S = {}; t0 = time.time()
    for p in PAIRS:
        v = dict(test=boot_sigma(L_TE, W_TE, *p, n), train=boot_sigma(L_TR, W_TR, *p, n), cond=boot_sigma(L_C, W_C, *p, min(n, len(W_C))),
                 unc=boot_sigma(L_M[:, :n], W_M[:n], *p, n), unc_rb=boot_sigma(L_M[:, :n], W_M[:n], *p, n, balance=True))
        if all(x is not None for x in v.values()): S[p] = v
    print(f"n={n}: {len(S)} pairs ({(time.time()-t0)/60:.0f} min)"); return S
S4 = location_set(4000); P4 = list(S4)
print(f"\n{'comparison (n=4000)':>36} {'SD of log ratio':>16} {'median ratio [class-block 95% CI]':>36}")
for nm, k in [('train vs test data (floor)', 'train'), ('conditional, uniform labels', 'cond'), ('unconditional, raw', 'unc'), ('unconditional, rebalanced', 'unc_rb')]:
    lr = np.array([np.log(S4[p][k]/S4[p]['test']) for p in P4]); ci = median_offset_ci(lr, P4)
    print(f"{nm:>36} {lr.std():16.3f} {np.exp(np.median(lr)):14.2f} [{ci[0]:.2f}, {ci[1]:.2f}]")
# residual offset of the rebalanced unconditional model at full size
n = len(W_M); lr_full, Pf = [], []
for p in PAIRS:
    sm = boot_sigma(L_M, W_M, *p, n, B=40, balance=True); sd = boot_sigma(L_TE, W_TE, *p, min(n, len(W_TE)), B=40)
    if sm and sd: lr_full.append(np.log(sm/sd)); Pf.append(p)
lr_full = np.array(lr_full); ci = median_offset_ci(lr_full, Pf)
print(f"\nunconditional rebalanced at n={n}: median ratio {np.exp(np.median(lr_full)):.2f}, geometric mean {np.exp(lr_full.mean()):.2f}, class-block 95% CI [{ci[0]:.2f}, {ci[1]:.2f}], SD {lr_full.std():.3f}")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].scatter([S4[p]['test'] for p in P4], [S4[p]['cond'] for p in P4], s=14, label='conditional (uniform labels)'); ax[0].scatter([S4[p]['test'] for p in P4], [S4[p]['unc'] for p in P4], s=14, label='unconditional, raw')
lim = [0.15, 5]; ax[0].plot(lim, lim, 'k--', lw=0.6); ax[0].set_xscale('log'); ax[0].set_yscale('log'); ax[0].set_xlabel('sigma_bif data'); ax[0].set_ylabel('sigma_bif model'); ax[0].legend(fontsize=7)
ax[1].scatter([DEC[p] for p in P], [S4[p]['unc'] if p in S4 else np.nan for p in P], s=14); ax[1].set_xscale('log'); ax[1].set_yscale('log'); ax[1].plot([0.1, 10], [0.1, 10], 'k--', lw=0.6); ax[1].set_xlabel('pair committor decision'); ax[1].set_ylabel('sigma_bif (model)')
plt.tight_layout(); plt.show()

## What to report back
- **Part 1:** the category lines, the sanity line, the two Spearman lines, the ratio lines.
- **Part 2:** the fidelity table and the floor.
- **Parts 3–4:** the location table and the full-size residual-offset line, plus both panels.